# Vigade analüüs

Selles märkmikus analüüsime `tagasiside_log.csv` logis leiduvaid halbu päringuid. Eesmärk on välja tuua juhtumid, millele süsteem vastas halvasti, ning arvutada iga vahesammu (metaandmete filtreerimine, RAGi vektorotsing, LLMi vastuse genereerimine) poolt tehtud vigade koguarv ja osakaal kõikidest vigadest.

In [ ]:
import pandas as pd

# 1. Loeme andmed
df = pd.read_csv('tagasiside_log.csv')

# 2. Eraldame negatiivse hinnanguga ('👎 Halb') päringud
df_bad = df[df['Hinnang'] == '👎 Halb'].copy()
print(f"Kokku leiti {len(df_bad)} vigast päringut.")

# Kuvame halvad juhtumid (kasutaja päring ja milles seisnes viga)
display(df_bad[['Aeg', 'Kasutaja päring', 'Veatüüp']])

## Vahesammude vigade koondtabel

Rakenduse vigade tüübid jagunevad kolmeks põhisammuks:
1. **Metaandmete filtreerimine** $\rightarrow$ *Filtrid olid liiga karmid/valed*
2. **RAGi vektorotsing** $\rightarrow$ *Otsing leidis valed ained (RAG viga)*
3. **LLMi vastuse genereerimine** $\rightarrow$ *LLM hallutsineeris/vastas valesti*

In [ ]:
# Seostame logitud veatüübid süsteemi vahesammudega
error_mapping = {
    'Filtrid olid liiga karmid/valed': 'Metaandmete filtreerimine',
    'Otsing leidis valed ained (RAG viga)': 'RAGi vektorotsing',
    'LLM hallutsineeris/vastas valesti': 'LLMi vastuse genereerimine'
}

df_bad['Vahesamm'] = df_bad['Veatüüp'].map(error_mapping).fillna('Määramata/Teadmata')

# Arvutame vigade koguarvud vastavalt vahesammule
error_counts = df_bad['Vahesamm'].value_counts().reset_index()
error_counts.columns = ['Vahesamm', 'Vigade koguarv']

# Lisame analüüsi kindlasti kõik kolm sammu isegi siis, kui nende väärtus on hetkel 0
all_steps = pd.DataFrame({
    'Vahesamm': ['Metaandmete filtreerimine', 'RAGi vektorotsing', 'LLMi vastuse genereerimine']
})

summary = pd.merge(all_steps, error_counts, on='Vahesamm', how='left').fillna(0)
summary['Vigade koguarv'] = summary['Vigade koguarv'].astype(int)

# Arvutame vigade osakaalu (protsendi)
total_errors = summary['Vigade koguarv'].sum()

if total_errors > 0:
    summary['Protsent (%)'] = (summary['Vigade koguarv'] / total_errors * 100).round(1)
else:
    summary['Protsent (%)'] = 0.0

display(summary)

# Vigade analüüs

Järgnev analüüs võtab kokku failis `tagasiside_log.csv` olevad halvad juhtumid ja arvutab iga vahesammu (metaandmete filtreerimine, RAGi vektorotsing, LLMi vastuse genereerimine) vigade koguarvu ja protsendi.

In [ ]:
import pandas as pd

# Loeme logid sisse
df = pd.read_csv('tagasiside_log.csv')

# Filtreerime välja halva hinnanguga juhtumid
bad_cases = df[df['Hinnang'] == '👎 Halb'].copy()
bad_cases.reset_index(drop=True, inplace=True)

print(f"Leiti {len(bad_cases)} halba juhtumit.")

## Halvad juhtumid (tabelina)

In [ ]:
# Kuvame halvad juhtumid. Lisame tabelisse olulised veerud.
display_cols = ['Kasutaja päring', 'Filtrid', 'Leitud ID-d', 'Veatüüp']
bad_cases[display_cols].style.set_properties(**{'text-align': 'left'})

## Vigade koondarvud ja protsendid

Veatüübid ja nende vastavus vahesammudele:
- **Filtrid olid liiga karmid/valed** -> Metaandmete filtreerimine
- **Otsing leidis valed ained (RAG viga)** -> RAGi vektorotsing
- **LLM hallutsineeris/vastas valesti** -> LLMi vastuse genereerimine

In [ ]:
total_bad = len(bad_cases)

error_mapping = {
    'Filtrid olid liiga karmid/valed': 'Metaandmete filtreerimine',
    'Otsing leidis valed ained (RAG viga)': 'RAGi vektorotsing',
    'LLM hallutsineeris/vastas valesti': 'LLMi vastuse genereerimine'
}

# Asendame logi tekstid kokkulepitud kategooriatega
bad_cases['Vahesamm'] = bad_cases['Veatüüp'].map(error_mapping).fillna('Teadmata/Määramata viga')

# Vastava vahesammu sageduse loendus
error_counts = bad_cases['Vahesamm'].value_counts().reset_index()
error_counts.columns = ['Vahesamm', 'Vigade koguarv']

# Lisame kõik oodatud kategooriad juhuks, kui mõnda viga pole kordagi esinenud
all_steps = pd.DataFrame({'Vahesamm': ['Metaandmete filtreerimine', 'RAGi vektorotsing', 'LLMi vastuse genereerimine']})
error_summary = pd.merge(all_steps, error_counts, on='Vahesamm', how='left').fillna(0)
error_summary['Vigade koguarv'] = error_summary['Vigade koguarv'].astype(int)

# Protsendi arvutus
if total_bad > 0:
    error_summary['Protsent (%)'] = (error_summary['Vigade koguarv'] / total_bad * 100).round(1)
else:
    error_summary['Protsent (%)'] = 0.0

error_summary